# LearnWise model server (Kaggle, free T4 GPU)

Runs **every model LearnWise needs** on Kaggle's GPU and gives you **one public URL** for both:

| What | Model | Used for |
|---|---|---|
| LLM | **Qwen3-8B** (4-bit, via Ollama) | answers, notes, quizzes, flashcards, mock exams, marking |
| Embeddings | **BAAI/bge-small-en-v1.5** (sentence-transformers) | searching your lectures (RAG) |
| Voice | **Kokoro-82M** (text-to-speech) | reading Ask answers aloud (optional: if it fails to load, everything else still works) |

Both sit behind a small gateway that needs a secret key, reached through a free **Cloudflare tunnel**.
Your laptop then needs no GPU and no PyTorch.

**Before you run:**
1. Right panel → *Session options* → **Accelerator: GPU T4 x2** (one T4 is enough).
2. **Internet: On** (needs a phone-verified Kaggle account).
3. Optional: *Add-ons → Secrets* → add `LEARNWISE_TOKEN` with any long random text, so your key stays the same between sessions.
4. **Run All**. After ~3-5 minutes section 7 prints a block of lines. Paste them into `backend/.env` and restart the backend.

**Keep in mind:** ~30 GPU hours/week per account, sessions end after 12 h or when idle, and the URL
**changes every restart** (paste the new block). Stop the session when you're done.

## 1. Settings

In [ ]:
MODEL = "qwen3:8b"                          # "qwen3:4b" = faster, "qwen3:14b" = better but slower (still fits a T4)
EMBED_MODEL = "BAAI/bge-small-en-v1.5"      # must match EMBEDDING_MODEL in backend/.env
CONTEXT_LENGTH = 8192                       # tokens the LLM can read at once; RAG prompts need room for passages
TUNNEL = "cloudflare"                       # "cloudflare" (no account) or "ngrok" (needs NGROK_AUTHTOKEN secret)
KEEP_ALIVE_HOURS = 8                        # how long the last cell keeps the session busy
GATEWAY_PORT = 8000
TTS_VOICE = "af_heart"                      # Kokoro voice that reads Ask answers aloud ("af_bella", "am_michael", "bf_emma", ...)

## 2. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

## 3. Secret key
Every request must carry this key, so someone who finds the URL can't use your GPU.

In [ ]:
import os, secrets
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("LEARNWISE_TOKEN")
    print("Using LEARNWISE_TOKEN from Kaggle Secrets")
except Exception:
    TOKEN = "lw-" + secrets.token_urlsafe(24)
    print("No LEARNWISE_TOKEN secret found, generated a new key for this session")

## 4. Install and start Ollama, download Qwen3 (~5 GB, 2-4 minutes)

In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess, time, requests

def wait_for(url, name, proc=None, log=None, max_wait=180):
    """Poll url until it answers. Any error (refused, slow, reset) just means "not ready yet"."""
    start = time.time()
    while time.time() - start < max_wait:
        try:
            if requests.get(url, timeout=10).ok:
                print(f"{name} is ready ({time.time() - start:.0f}s)")
                return True
        except requests.RequestException:
            pass
        if proc is not None and proc.poll() is not None:
            raise RuntimeError(f"{name} exited. Last log lines:\n" + (open(log).read()[-3000:] if log else ""))
        print(f"  waiting for {name}... {time.time() - start:.0f}s", end="\r", flush=True)
        time.sleep(3)
    raise RuntimeError(f"{name} did not answer within {max_wait}s." + (f" Log: {log}" if log else ""))

def answers(url):
    try:
        return requests.get(url, timeout=10).ok
    except requests.RequestException:
        return False

# Re-running this cell is safe: if Ollama is already up (from an earlier run or the installer), reuse it.
if answers("http://localhost:11434/api/version"):
    ollama = None
    print("Ollama is already running, reusing it")
else:
    subprocess.run("pkill -f 'ollama serve'", shell=True)  # a half-started old copy would block the port
    time.sleep(2)
    env = dict(os.environ,
               OLLAMA_HOST="127.0.0.1:11434",            # only the gateway talks to Ollama
               OLLAMA_KEEP_ALIVE="-1",                   # keep the model loaded in GPU memory
               OLLAMA_CONTEXT_LENGTH=str(CONTEXT_LENGTH),
               OLLAMA_NUM_PARALLEL="2")                  # you and Sana can send requests at the same time
    ollama = subprocess.Popen(["ollama", "serve"], env=env,
                              stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
    wait_for("http://localhost:11434/api/version", "Ollama", ollama, "/kaggle/working/ollama.log")

In [ ]:
!ollama pull {MODEL}

In [ ]:
# First call loads the model into GPU memory (can take 1-2 minutes); retry instead of failing.
for attempt in range(1, 4):
    t = time.time()
    try:
        r = requests.post("http://localhost:11434/v1/chat/completions", timeout=600, json={
            "model": MODEL, "max_tokens": 100,
            "messages": [{"role": "user", "content": "In one sentence, what is retrieval-augmented generation? /no_think"}],
        })
        r.raise_for_status()
        print(r.json()["choices"][0]["message"]["content"])
        print(f"took {time.time() - t:.1f}s (the first call also loads the model)")
        break
    except requests.RequestException as e:
        print(f"attempt {attempt} failed after {time.time() - t:.0f}s: {e}")
        time.sleep(10)
else:
    raise RuntimeError("Qwen3 did not answer. Check /kaggle/working/ollama.log and that the GPU is on.")

## 5. Start the gateway (LLM + embeddings + voice on one port)

In [ ]:
%%writefile /kaggle/working/gateway.py
"""LearnWise gateway: one OpenAI-compatible URL for everything the app needs.

  GET  /health                 no auth, for quick checks
  GET  /v1/models              chat model(s) from Ollama + the embedding model
  POST /v1/chat/completions    forwarded to Ollama (Qwen3)
  POST /v1/embeddings          bge-small-en-v1.5 on the GPU (sentence-transformers)
  POST /v1/audio/speech        Kokoro-82M text-to-speech, returns a WAV file (OpenAI-style request)

Every /v1 call needs  Authorization: Bearer <LEARNWISE_TOKEN>  so a leaked URL is useless.
"""
import io
import os
import threading
import time

import numpy as np

import httpx
import torch
from fastapi import FastAPI, Request
from fastapi.concurrency import run_in_threadpool
from fastapi.responses import JSONResponse, Response
from sentence_transformers import SentenceTransformer

OLLAMA = "http://localhost:11434"
TOKEN = os.environ["LEARNWISE_TOKEN"]
EMBED_MODEL = os.environ.get("EMBED_MODEL", "BAAI/bge-small-en-v1.5")
TTS_VOICE = os.environ.get("TTS_VOICE", "af_heart")
TTS_MAX_CHARS = 4000

device = "cuda" if torch.cuda.is_available() else "cpu"
embedder = SentenceTransformer(EMBED_MODEL, device=device)

# Voice is optional: if Kokoro can't load, the LLM and embeddings keep working and /v1/audio/speech says why.
tts, tts_error, tts_lock = None, "", threading.Lock()
try:
    import soundfile as sf
    from kokoro import KPipeline
    try:
        tts = KPipeline(lang_code=TTS_VOICE[0], repo_id="hexgrad/Kokoro-82M", device=device)  # 'a' = American English
    except TypeError:  # older kokoro without the device argument
        tts = KPipeline(lang_code=TTS_VOICE[0])
except Exception as e:  # missing package, download failure, ...
    tts_error = f"{type(e).__name__}: {e}"
app = FastAPI(title="LearnWise gateway")
client = httpx.AsyncClient(timeout=600)


@app.middleware("http")
async def auth(request: Request, call_next):
    if request.url.path.startswith("/v1") and request.headers.get("authorization", "") != f"Bearer {TOKEN}":
        return JSONResponse({"error": {"message": "Missing or wrong API key (LLM_API_KEY / EMBEDDING_API_KEY in backend/.env)."}}, status_code=401)
    return await call_next(request)


@app.get("/health")
async def health():
    try:
        ollama_ok = (await client.get(f"{OLLAMA}/api/version", timeout=5)).status_code == 200
    except httpx.HTTPError:
        ollama_ok = False
    return {"ok": ollama_ok, "embedding_model": EMBED_MODEL, "device": device,
            "tts": tts is not None, "tts_voice": TTS_VOICE if tts else None, "tts_error": tts_error or None}


@app.get("/v1/models")
async def models():
    r = await client.get(f"{OLLAMA}/v1/models")
    data = r.json()
    data.setdefault("data", []).append({"id": EMBED_MODEL, "object": "model", "owned_by": "sentence-transformers"})
    if tts is not None:
        data["data"].append({"id": "kokoro", "object": "model", "owned_by": "hexgrad"})
    return data


@app.post("/v1/chat/completions")
async def chat(request: Request):
    r = await client.post(f"{OLLAMA}/v1/chat/completions", content=await request.body(), headers={"Content-Type": "application/json"})
    return Response(r.content, status_code=r.status_code, media_type="application/json")


@app.post("/v1/embeddings")
async def embeddings(body: dict):
    texts = body.get("input", [])
    texts = [texts] if isinstance(texts, str) else list(texts)
    if not texts or len(texts) > 256:
        return JSONResponse({"error": {"message": "input must be 1-256 strings"}}, status_code=400)
    t = time.time()
    vecs = await run_in_threadpool(embedder.encode, texts, batch_size=64, normalize_embeddings=True, show_progress_bar=False)
    return {
        "object": "list",
        "model": EMBED_MODEL,
        "data": [{"object": "embedding", "index": i, "embedding": v.tolist()} for i, v in enumerate(vecs)],
        "usage": {"prompt_tokens": sum(len(x.split()) for x in texts), "total_tokens": sum(len(x.split()) for x in texts), "seconds": round(time.time() - t, 3)},
    }


def _speak(text: str, voice: str, speed: float) -> bytes:
    """Kokoro splits long text into segments; join their audio into one 24 kHz WAV."""
    with tts_lock:  # one pipeline, one request at a time
        parts = [np.asarray(audio.cpu() if hasattr(audio, "cpu") else audio, dtype=np.float32)
                 for _, _, audio in tts(text, voice=voice, speed=speed) if audio is not None]
    buf = io.BytesIO()
    sf.write(buf, np.concatenate(parts) if parts else np.zeros(2400, dtype=np.float32), 24000, format="WAV")
    return buf.getvalue()


@app.post("/v1/audio/speech")
async def speech(body: dict):
    if tts is None:
        return JSONResponse({"error": {"message": f"Voice model not loaded: {tts_error or 'unknown error'}"}}, status_code=503)
    text = str(body.get("input", "")).strip()
    if not text or len(text) > TTS_MAX_CHARS:
        return JSONResponse({"error": {"message": f"input must be 1-{TTS_MAX_CHARS} characters"}}, status_code=400)
    voice = str(body.get("voice") or TTS_VOICE)
    speed = min(2.0, max(0.5, float(body.get("speed") or 1.0)))
    t = time.time()
    try:
        wav = await run_in_threadpool(_speak, text, voice, speed)
    except Exception as e:  # e.g. unknown voice name
        return JSONResponse({"error": {"message": f"Speech failed: {type(e).__name__}: {e}"}}, status_code=400)
    return Response(wav, media_type="audio/wav", headers={"X-Seconds": f"{time.time() - t:.2f}"})

In [ ]:
!pip -q install "sentence-transformers>=3.0" fastapi "uvicorn[standard]" httpx "kokoro>=0.9.4" soundfile 2>&1 | tail -1
!sudo apt-get -qq install -y espeak-ng > /dev/null   # Kokoro's fallback for words it doesn't know

subprocess.run(f"pkill -f 'uvicorn gateway:app'", shell=True)  # safe to re-run this cell
time.sleep(2)
gw_env = dict(os.environ, LEARNWISE_TOKEN=TOKEN, EMBED_MODEL=EMBED_MODEL, TTS_VOICE=TTS_VOICE)
gateway = subprocess.Popen(["python", "-m", "uvicorn", "gateway:app", "--host", "127.0.0.1", "--port", str(GATEWAY_PORT)],
                           cwd="/kaggle/working", env=gw_env,
                           stdout=open("/kaggle/working/gateway.log", "w"), stderr=subprocess.STDOUT)
# first start downloads bge-small (~130 MB) and Kokoro (~330 MB)
wait_for(f"http://localhost:{GATEWAY_PORT}/health", "Gateway", gateway, "/kaggle/working/gateway.log", max_wait=600)
print(requests.get(f"http://localhost:{GATEWAY_PORT}/health", timeout=10).json())

AUTH = {"Authorization": f"Bearer {TOKEN}"}
e = requests.post(f"http://localhost:{GATEWAY_PORT}/v1/embeddings", headers=AUTH, timeout=120,
                  json={"model": EMBED_MODEL, "input": ["gradient descent", "how weights are updated"]}).json()
v = e["data"]
print("embedding size:", len(v[0]["embedding"]), "| similarity:", round(sum(a * b for a, b in zip(v[0]["embedding"], v[1]["embedding"])), 3))

t = time.time()
sp = requests.post(f"http://localhost:{GATEWAY_PORT}/v1/audio/speech", headers=AUTH, timeout=120,
                   json={"model": "kokoro", "input": "Photosynthesis converts light energy into chemical energy."})
if sp.ok:
    print(f"voice: OK, {len(sp.content) / 48000:.1f} s of audio in {time.time() - t:.1f} s")
    from IPython.display import Audio, display
    display(Audio(sp.content))  # press play to hear it
else:
    print("voice: OFF (the app will use the browser's voice instead):", sp.text[:300])

## 6. Open the public tunnel

In [ ]:
import re

CF_BIN, CF_LOG = "/kaggle/working/cloudflared", "/kaggle/working/cloudflared.log"

def start_cloudflare(wait_s=90):
    """Start a quick tunnel and return its URL, or None. Never blocks longer than wait_s."""
    if not os.path.exists(CF_BIN):
        subprocess.run(f"wget -q -O {CF_BIN} https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
                       f" && chmod +x {CF_BIN}", shell=True, check=True)
    subprocess.run("pkill -f 'cloudflared tunnel'", shell=True)  # safe to re-run this cell
    time.sleep(1)
    # --protocol http2: Kaggle often blocks the UDP (QUIC) connection cloudflared tries first, which makes it hang
    subprocess.Popen([CF_BIN, "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", f"http://localhost:{GATEWAY_PORT}"],
                     stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT)
    start = time.time()
    while time.time() - start < wait_s:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
        if m and "api.trycloudflare.com" not in m.group(0):
            return m.group(0)
        print(f"  waiting for Cloudflare... {time.time() - start:.0f}s", end="\r", flush=True)
        time.sleep(2)
    print("\nCloudflare gave no URL. Last log lines:\n" + open(CF_LOG).read()[-2000:])
    return None

def start_ngrok():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")  # raises if the secret isn't set
    subprocess.run("pip -q install pyngrok", shell=True, check=True)
    from pyngrok import ngrok
    ngrok.set_auth_token(token)
    return ngrok.connect(GATEWAY_PORT, "http").public_url

public_url = None
if TUNNEL == "cloudflare":
    for attempt in range(1, 4):
        public_url = start_cloudflare()
        if public_url:
            break
        print(f"attempt {attempt} failed, retrying...")
if not public_url:
    try:
        print("Trying ngrok instead...")
        public_url = start_ngrok()
    except Exception as e:
        raise RuntimeError(
            "No tunnel could be opened. Fix: get a free token at https://dashboard.ngrok.com/get-started/your-authtoken, "
            "add it in Add-ons -> Secrets as NGROK_AUTHTOKEN, set TUNNEL = \"ngrok\" in section 1 and re-run from here. "
            f"(ngrok error: {e})") from None
print("\npublic URL:", public_url)

## 7. Test through the tunnel and print your `.env` lines

In [ ]:
base = f"{public_url}/v1"
wait_for(f"{public_url}/health", "Tunnel", max_wait=120)  # a new Cloudflare URL can take a few seconds
t = time.time()
chat = requests.post(f"{base}/chat/completions", headers=AUTH, timeout=180, json={
    "model": MODEL, "max_tokens": 30, "messages": [{"role": "user", "content": "Say hello in 5 words. /no_think"}]})
print("chat:", chat.status_code, chat.json()["choices"][0]["message"]["content"].strip(), f"({time.time() - t:.1f}s)")
emb = requests.post(f"{base}/embeddings", headers=AUTH, timeout=60, json={"model": EMBED_MODEL, "input": ["hello"]})
print("embeddings:", emb.status_code, "dim", len(emb.json()["data"][0]["embedding"]))
sp = requests.post(f"{base}/audio/speech", headers=AUTH, timeout=120, json={"model": "kokoro", "input": "Hello from LearnWise."})
print("voice:", sp.status_code, "OK" if sp.ok else sp.text[:200])
print("without key:", requests.get(f"{base}/models", timeout=30).status_code, "(401 = protected)")

ENV_BLOCK = f"""USE_MOCKS=false
MOCK_RETRIEVAL=false
LLM_BASE_URL={base}
LLM_MODEL={MODEL}
LLM_API_KEY={TOKEN}
EMBEDDING_MODEL={EMBED_MODEL}
EMBEDDING_BASE_URL={base}
EMBEDDING_API_KEY={TOKEN}"""
if sp.ok:  # leave these out when voice failed: the app then uses the browser's voice
    ENV_BLOCK += f"""
TTS_BASE_URL={base}
TTS_API_KEY={TOKEN}"""
open("/kaggle/working/learnwise.env", "w").write(ENV_BLOCK + "\n")
print("\n" + "=" * 72)
print("Paste these lines into backend/.env (replace the old ones), then restart the backend:\n")
print(ENV_BLOCK)
print("=" * 72)
print("Also saved as /kaggle/working/learnwise.env (Output panel). Keep the key private.")

## 8. Keep the session alive

Leave this cell running while you work. Interrupt it (■) and **stop the session** when you are done.
If the tunnel dies, re-run sections 6 and 7 and paste the new lines.

In [ ]:
end = time.time() + KEEP_ALIVE_HOURS * 3600
while time.time() < end:
    try:
        ok = requests.get(f"{public_url}/v1/models", headers=AUTH, timeout=15).status_code == 200
    except Exception:
        ok = False
    alive = answers("http://localhost:11434/api/version") and gateway.poll() is None
    print(time.strftime("%H:%M"), "tunnel OK" if ok else "tunnel NOT reachable", "| servers", "running" if alive else "STOPPED", "|", public_url, flush=True)
    time.sleep(300)